# 04. Eleição do vencedor e promoção controlada

1. **Vencedor do ciclo:** menor MAE na validação entre A e B. Em empate fica A, o mais simples.
2. **Vigente:** o modelo apontado por `models/modelo_vigente.json` é carregado do MLflow e avaliado na mesma janela de validação, com as features dele.
3. **Regra de promoção (`config.yaml`):** o vencedor precisa errar menos que a persistência e pelo menos 2% menos que o vigente. No primeiro ciclo, sem vigente, basta bater a persistência.
4. **Resultado:** se passar, ele vira o vigente. Se não, o vigente continua. Nos dois casos a decisão vai para o manifesto e para o MLflow, e os modelos anteriores continuam no MLflow.

A comparação é justa porque o vigente só treinou com dias anteriores à validação de qualquer ciclo seguinte (as janelas só andam para frente). O código confere isso antes de comparar e para o pipeline se não for verdade.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from src import ciclo, modelagem, preprocessamento, promocao

cfg = ciclo.carregar_config()
estado = ciclo.atual()
mlflow = ciclo.configurar_mlflow(cfg)
dataset, janelas = preprocessamento.carregar(cfg)
pd.set_option("display.precision", 3)
print(f"ciclo {estado['cycle_id']} | {estado['data_version']}")
from datetime import datetime, timezone

In [ ]:
metrica, direcao = cfg["avaliacao"]["metrica_principal"], cfg["avaliacao"]["direcao"]
comp = estado["competicao"]
vencedor = promocao.eleger({p: comp[p]["validacao"] for p in cfg["modelos"]}, metrica, direcao)
desafiante = comp[vencedor]
print(f"vencedor do ciclo: modelo {vencedor} ({desafiante['algoritmo']}), {metrica} = {desafiante['validacao'][metrica]:.3f}")

In [ ]:
validacao = janelas["validacao"]
manifesto = promocao.ler_manifesto()
run_id_vigente = manifesto["run_id"] if manifesto else None
valor_vigente = None
if manifesto:
    if manifesto["treino_ate"] >= str(validacao.index.min().date()):
        raise RuntimeError("o vigente treinou com dias da validação atual: a comparação não seria justa")
    vigente = mlflow.sklearn.load_model(manifesto["model_uri"])
    m_vigente = modelagem.metricas(validacao.alvo, vigente.predict(validacao[manifesto["features"]]))
    valor_vigente = m_vigente[metrica]
    print(f"vigente: {manifesto['algoritmo']} do ciclo {manifesto['cycle_id']}, {metrica} = {valor_vigente:.3f} na mesma janela")
else:
    print("ainda não existe modelo vigente")

promover, motivo = promocao.decidir(desafiante["validacao"][metrica], valor_vigente, comp["persistencia"][metrica], cfg)
decisao = "promovido" if promover else "mantido"
print(f"decisão: {decisao} ({motivo})")

## Manifesto do modelo vigente

Se houve promoção, o manifesto passa a apontar para o desafiante, e o vigente antigo vai para a lista `anteriores`. Se não houve, só `ultima_decisao` muda.

In [ ]:
registro = {"cycle_id": estado["cycle_id"], "data_version": estado["data_version"], "decisao": decisao,
            "motivo": motivo, "vencedor": vencedor, "algoritmo_vencedor": desafiante["algoritmo"],
            "run_id_vencedor": desafiante["run_id"], "run_id_vigente": run_id_vigente,
            f"{metrica}_vencedor": desafiante["validacao"][metrica],
            f"{metrica}_vigente": valor_vigente, f"{metrica}_persistencia": comp["persistencia"][metrica],
            "limiar_melhoria": cfg["promocao"]["limiar_melhoria"], "janela_validacao": comp["janela_validacao"]}

if promover:
    anteriores = (manifesto or {}).get("anteriores", [])
    if manifesto:
        anteriores.append({k: manifesto[k] for k in ("cycle_id", "algoritmo", "run_id", "model_uri", "promovido_em")})
    manifesto = {"cycle_id": estado["cycle_id"], "data_version": estado["data_version"], "papel": vencedor,
                 "algoritmo": desafiante["algoritmo"], "run_id": desafiante["run_id"], "model_uri": desafiante["model_uri"],
                 "features": estado["votacao"]["features"], "metrica": metrica,
                 "valor_validacao": desafiante["validacao"][metrica], "treino_ate": comp["treino_ate"],
                 "promovido_em": datetime.now(timezone.utc).isoformat(timespec="seconds"), "anteriores": anteriores}
if manifesto:
    manifesto["ultima_decisao"] = registro
    promocao.gravar_manifesto(manifesto)
manifesto

In [ ]:
tags = ciclo.tags(estado, "decisao") | {"decisao": decisao, "vencedor": vencedor}
with mlflow.start_run(run_name=f"{estado['cycle_id']}-decisao", tags=tags) as run:
    mlflow.log_params({"metrica": metrica, "direcao": direcao, "limiar_melhoria": cfg["promocao"]["limiar_melhoria"],
                       "exigir_bater_persistencia": cfg["promocao"]["exigir_bater_persistencia"], "motivo": motivo,
                       "run_id_vencedor": desafiante["run_id"], "run_id_vigente": run_id_vigente})
    mlflow.log_metrics({k: v for k, v in registro.items() if k.startswith(metrica) and v is not None}
                       | {"promovido": int(promover)})
    if manifesto:
        mlflow.log_dict(manifesto, "modelo_vigente.json")

estado["decisao"] = registro | {"run_id": run.info.run_id}
ciclo.salvar(estado)